<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 5 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Batch loading, failures, and retries</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">Load historical business tables and new orders to practice response checks, safe retries, and error handling.</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">Target Doris 4.1.3 · Order data · Isolated lab database</span>
</div>

First load ten simulated new orders with Stream Load to practice retries and error handling, then extend to 10 WWI historical tables (701,846 rows). Run in order.

[Reading](course5_batch_and_streaming_ingestion.md) · [Course home](../README.md)


## Lab scope

Rebuild only orders_imported and wwi_orders, wwi_order_lines, wwi_customers, wwi_products, wwi_invoices, wwi_invoice_lines, wwi_customer_transactions, wwi_payment_methods, wwi_transaction_types, wwi_delivery_methods.

The historical Parquet archive and simulated CSV are downloaded from the course object-storage bucket and validated on first use. See the [Data guide](../../datasets/README.md). The independent exercise separately rebuilds orders_mapping_practice. Course tools automatically configure the sandbox's BE HTTP address. This lab practices Stream Load; see the reading for other ingestion paths and how they work.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import dataset_path, fixture, expect, normalized
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows
from dw_course.ui import show_sql, show_response

lab = connect_sandbox()


from uuid import uuid4



### Start by ingesting ten new orders

Order IDs 900001–900010 come from COURSE_SIMULATION and total 1400.00. Use this single CSV to understand request parameters and results before extending to WWI historical tables.


## 1. Load simulated new orders

Load one batch at a time using off_mode. First inspect Status and the loaded and filtered row counts in the response, then query the target table to verify ten orders totaling 1400.00. HTTP success is only a transport-layer result; the returned transaction status determines whether loading completed.

This time, inspect the full HTTP request rather than calling a wrapper. Read the code in this order:

1. The URL points to orders_imported in the current lab database; the connection step supplies the BE address.
2. headers describes the CSV format, column order, and batch label; columns comes from ORDER_COLUMNS imported earlier.
3. open(..., "rb") reads raw file bytes, and requests.put sends them as the request body; auth uses existing connection credentials without printing the password.
4. raise_for_status checks HTTP errors, response.json() reads the Doris load result, and separate checks verify status, row counts, and amounts in the table.

The local sandbox accesses the BE directly without following redirects. Do not change the URL to the FE address.
Keep the label for the next retry step; do not simply rerun the request with a new label, or the data will be appended again.


In [ ]:
import os
import requests

orders_csv = dataset_path("orders.csv")  # Validate before resetting the target table

lab.execute("DROP TABLE IF EXISTS orders_imported")
ddl = order_ddl("orders_imported")
show_sql("Table creation SQL", ddl)
lab.execute(ddl)
label = "dw_l1_" + uuid4().hex
columns = ",".join(ORDER_COLUMNS)
url = os.environ["DW_BE_HTTP_URL"] + f"/api/{lab.database}/orders_imported/_stream_load"
headers = {
    "label": label, "format": "csv", "column_separator": ",",
    "columns": columns, "strict_mode": "true", "max_filter_ratio": "0",
    "group_commit": "off_mode",
}
with orders_csv.open("rb") as payload:
    response = requests.put(url, auth=(lab.user, lab.password),
                            headers=headers, data=payload, timeout=120,
                            allow_redirects=False)
response.raise_for_status()
result = response.json()
show_response(result)
expect(result["Status"], "Success")
expect(result["NumberLoadedRows"], 10)
expect(result["NumberFilteredRows"], 0)
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_imported"), [(10,"1400.00")])
lab.sql("SELECT order_id, customer_id, order_amount FROM orders_imported ORDER BY order_id", title="Ten loaded new orders")


## 2. Retry the same batch with the same label

Immediately resend the same file and label. Observe Doris recognizing the existing batch, leaving ten rows in the target table. A label identifies a load batch within its retention period; a new label is treated as another batch request. Module 7 uses business keys and versions to handle duplicate events.

The lab.stream_load below wraps the preceding HTTP request, reusing the same address, authentication, file, column mapping, and quality parameters.
It returns parsed JSON; reusing the original label here does not initiate a new load batch.


In [ ]:
retry = lab.stream_load("orders_imported", dataset_path("orders.csv"), label, columns)
show_response(retry)
expect(retry["Status"], "Label Already Exists")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_imported"), [(10,"1400.00")])


## 3. Observe whole-batch rejection of bad data

Load two rows with a new label, one with an amount that cannot be converted. With strict_mode=true and max_filter_ratio=0, expect the entire batch to fail, leaving the original ten rows unchanged.


In [ ]:
rejected = lab.stream_load("orders_imported", dataset_path("malformed_orders.csv"),
                           "dw_bad_" + uuid4().hex, columns)
show_response(rejected)
expect(rejected["Status"], "Fail")
expect(rejected["NumberFilteredRows"], 1)
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_imported"), [(10,"1400.00")])
lab.sql("SELECT COUNT(*) AS orders, SUM(order_amount) AS amount FROM orders_imported", title="Original data remains intact after rejection of the invalid batch")


## 4. Extend to ten historical business tables

After loading a single file, use the same request method to batch-load historical Parquet. Validate all files against the course manifest before rebuilding tables. Read the structures of Orders and OrderLines, and Invoices and InvoiceLines, to distinguish business tables from line-item tables.


In [ ]:
from dw_course.wwi import manifest, parquet_paths, parquet_ddl
paths = parquet_paths()  # Missing files or checksum mismatches stop execution before tables are dropped
for name, metadata in manifest()["tables"].items():
    target = "wwi_" + name
    lab.execute("DROP TABLE IF EXISTS " + target)
    ddl = parquet_ddl(name, target)
    show_sql("WWI historical table: " + name, ddl)
    lab.execute(ddl)
    response = lab.stream_load(target, paths[name], "wwi_" + uuid4().hex, format="parquet")
    show_response(response)
    expect(response["Status"], "Success")
    expect(response["NumberLoadedRows"], metadata["rows"])
    expect(response["NumberFilteredRows"], 0)
    primary = metadata["primary"]
    expect(lab.query(f"SELECT COUNT(*), COUNT(DISTINCT {primary}) FROM {target}"),
           [(metadata["rows"], metadata["rows"])])


### After a successful load, check orders and lines first

Check whether joining order lines to orders, customers, and products multiplies or loses rows, then summarize historical orders by date.
This section does not require mastery of the complete accounting model; SQL for invoices and account receipts is in the [optional further reading](optional_invoice_and_receipts.md).


In [ ]:
expect(lab.query("""
SELECT COUNT(*), SUM(l.Quantity*l.UnitPrice)
FROM wwi_order_lines l
JOIN wwi_orders o ON l.OrderID=o.OrderID
JOIN wwi_customers c ON o.CustomerID=c.CustomerID
JOIN wwi_products p ON l.StockItemID=p.StockItemID
"""), [(231412, "177634276.40")])
lab.sql("""
SELECT o.OrderDate, COUNT(DISTINCT o.OrderID) AS orders,
       SUM(l.Quantity*l.UnitPrice) AS order_amount
FROM wwi_orders o JOIN wwi_order_lines l ON o.OrderID=l.OrderID
GROUP BY o.OrderDate ORDER BY o.OrderDate LIMIT 10
""", title="Daily order analysis across the full history")


## Completion and troubleshooting

When loading fails, retain the response and promptly inspect ErrorURL in the trusted lab environment to identify error rows and causes. For Publish Timeout, confirm the final outcome using the original label and transaction details before deciding to retry. Module 6 creates tables for long-term retention of raw input and rejection reasons.


## Your turn and verification

Query WWI product rankings by sales amount and confirm that each line is counted once. The optional further reading compares invoice amounts with customer account receipts; it is not required to complete this lab.

After this section, continue to Module 6 to check customer references and quality for new orders, then Module 7 to handle payment, refund, and delivery events.


## Independent exercise

`dataset_path("orders_reordered.csv")` contains two new orders. Its first two columns are customer_id and order_id; the remaining columns follow the original CSV order. Independently create orders_mapping_practice and specify the correct columns through HTTP Stream Load. Verify that order 901001 belongs to customer 1 with amount 180, and order 901002 belongs to customer 2 with amount 80.

Follow the HTTP example in step 1, using a new target table, file, and label, and swapping the first two fields in columns. Fill in the URL, request headers, and column mapping yourself; read the password from the existing connection object rather than writing it in the notebook. This exercise rebuilds only orders_mapping_practice.

Write and run your code in the next cell, then expand the reference solution after finishing. A blank exercise is not automatically marked complete.


In [ ]:
# Write your SQL or load request here.


<details>
<summary>Reference solution (expand after completing the exercise)</summary>

```python
import os
import requests
lab.execute("DROP TABLE IF EXISTS orders_mapping_practice")
ddl = order_ddl("orders_mapping_practice")
show_sql("Exercise table structure", ddl)
lab.execute(ddl)
# The first two input columns are customer ID and order ID; the table uses a different field order.
practice_columns = "customer_id,order_id,order_amount,status,event_version,event_id,event_time,paid_amount,refund_amount,region,data_source"
practice_url = os.environ["DW_BE_HTTP_URL"] + f"/api/{lab.database}/orders_mapping_practice/_stream_load"
practice_headers = {
    "label": "mapping_" + uuid4().hex, "format": "csv", "column_separator": ",",
    "columns": practice_columns, "strict_mode": "true", "max_filter_ratio": "0",
    "group_commit": "off_mode",
}
with (dataset_path("orders_reordered.csv")).open("rb") as payload:
    response = requests.put(practice_url, auth=(lab.user, lab.password),
                            headers=practice_headers, data=payload, timeout=120,
                            allow_redirects=False)
response.raise_for_status()
result = response.json()
show_response(result, title="Independent column-mapping load result")
expect(result["Status"], "Success")
lab.sql("SELECT order_id, customer_id, order_amount FROM orders_mapping_practice ORDER BY order_id", title="Order ID and customer ID mapping results")
expect(lab.query("SELECT order_id, customer_id, order_amount FROM orders_mapping_practice ORDER BY order_id"),
       [(901001,1,"180.00"),(901002,2,"80.00")])
```

</details>
